# Regional Food Price Forecasts

Builds regional monthly food-price time series from `somalia/data/wfp_food_prices_som.csv` and forecasts each region with the local seasonal baseline.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'project').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project.helper.predictions import (
    list_wfp_food_regions,
    load_regional_food_prices,
    predict_regional_food_price,
    predict_regional_food_prices,
)


## Load Test

In [2]:
regions = list_wfp_food_regions()
print('regions:', len(regions), regions)

gedo_history = load_regional_food_prices('Gedo')
assert len(regions) > 0
assert len(gedo_history) > 0
assert gedo_history['value'].isna().sum() == 0

gedo_history.tail()


regions: 18 ['Awdal', 'Bakool', 'Banadir', 'Bari', 'Bay', 'Galgaduud', 'Gedo', 'Hiraan', 'Juba Dhexe', 'Juba Hoose', 'Mudug', 'Nugaal', 'Sanaag', 'Shabelle Dhexe', 'Shabelle Hoose', 'Sool', 'Togdheer', 'Woqooyi Galbeed']
[predictions] Extended Gedo food series with seasonal monthly median: 2026-05-01 to 2026-05-01.
[predictions] Loaded Gedo regional food price proxy: 2017-04-01 to 2026-05-01, 110 rows, aggregation=median, missing 9->0.


,date,value,source,region
105,2026-01-01,50.83,observed,Gedo
106,2026-02-01,51.14,observed,Gedo
107,2026-03-01,51.58,observed,Gedo
108,2026-04-01,59.38,observed,Gedo
109,2026-05-01,45.19,seasonal_extension,Gedo


## Single Region Forecast

In [3]:
gedo_prediction = predict_regional_food_price('Gedo', source='local')

assert len(gedo_prediction.history) > 0
assert len(gedo_prediction.forecast) > 0

gedo_prediction.forecast


[predictions] Extended Gedo food series with seasonal monthly median: 2026-05-01 to 2026-05-01.
[predictions] Loaded Gedo regional food price proxy: 2017-04-01 to 2026-05-01, 110 rows, aggregation=median, missing 9->0.
[predictions] Building local seasonal forecast: gedo_food_sybilion
[predictions] Saved prediction plot: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/somalia/data/sybilion_regional_food/gedo/gedo_food_sybilion_forecast.png


,date,forecast,q05,q10,q25,q50,q75,q90,q95
0,2026-06-01,43.067222,38.76050,38.76050,40.913861,43.067222,45.220583,47.373944,47.373944
1,2026-07-01,40.348056,36.31325,36.31325,38.330653,40.348056,42.365458,44.382861,44.382861
2,2026-08-01,40.923889,36.83150,36.83150,38.877694,40.923889,42.970083,45.016278,45.016278
3,2026-09-01,42.833056,38.54975,38.54975,40.691403,42.833056,44.974708,47.116361,47.116361
4,2026-10-01,43.385556,39.04700,39.04700,41.216278,43.385556,45.554833,47.724111,47.724111
5,2026-11-01,44.558889,40.10300,40.10300,42.330944,44.558889,46.786833,49.014778,49.014778


## All Region Forecasts

In [4]:
regional_predictions = predict_regional_food_prices(source='local')
assert len(regional_predictions) == len(regions)

summary_rows = []
for region_key, result in regional_predictions.items():
    assert result.history['value'].isna().sum() == 0
    assert result.forecast['forecast'].notna().all()
    summary_rows.append(
        {
            'region': region_key,
            'history_rows': len(result.history),
            'forecast_rows': len(result.forecast),
            'last_observed': result.history['date'].max(),
            'next_forecast': result.forecast['date'].min(),
            'forecast_mean': result.forecast['forecast'].mean(),
        }
    )

summary = pd.DataFrame(summary_rows).sort_values('region')
summary


[predictions] Extended Awdal food series with seasonal monthly median: 2020-04-01 to 2026-05-01.
[predictions] Loaded Awdal regional food price proxy: 1996-01-01 to 2026-05-01, 365 rows, aggregation=median, missing 48->0.
[predictions] Building local seasonal forecast: awdal_food_sybilion
[predictions] Saved prediction plot: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/somalia/data/sybilion_regional_food/awdal/awdal_food_sybilion_forecast.png
[predictions] Extended Bakool food series with seasonal monthly median: 2026-05-01 to 2026-05-01.
[predictions] Loaded Bakool regional food price proxy: 1999-11-01 to 2026-05-01, 319 rows, aggregation=median, missing 36->0.
[predictions] Building local seasonal forecast: bakool_food_sybilion
[predictions] Saved prediction plot: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/somalia/data/sybilion_regional_food/bakool/bakool_food_sybilion_forecast.png
[predictions] Extended Banadir food series with seasonal monthly median: 2026-05-01 to 2026-05-01.
[predictio

,region,history_rows,forecast_rows,last_observed,next_forecast,forecast_mean
0,awdal,365,6,2026-05-01,2026-06-01,2.917802
1,bakool,319,6,2026-05-01,2026-06-01,20.176643
2,banadir,377,6,2026-05-01,2026-06-01,12.451314
3,bari,371,6,2026-05-01,2026-06-01,26.400382
4,bay,371,6,2026-05-01,2026-06-01,11.680967
5,galgaduud,110,6,2026-05-01,2026-06-01,39.886944
6,gedo,110,6,2026-05-01,2026-06-01,42.519444
7,hiraan,332,6,2026-05-01,2026-06-01,16.373491
8,juba_dhexe,351,6,2026-05-01,2026-06-01,10.269501
9,juba_hoose,365,6,2026-05-01,2026-06-01,16.006891
